In [19]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory():
    return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

csv_file = "results/ResNet18_metrics.csv"

if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])

method_name = "linear_probe"


Using device: cuda


In [20]:

# Pretrained ResNet50 weights
weights = ResNet18_Weights.IMAGENET1K_V1

# Use ResNet preprocessing
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)





# Split train/val
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))



Train samples: 68175
Val samples: 7575
Test samples: 25250


In [21]:
## Linear Probing with ResNet

# Load pretrained ResNet
model = models.resnet18(weights=weights)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 101)

# Unfreeze FC layer
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

print(model.fc)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total



Linear(in_features=512, out_features=101, bias=True)


In [22]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # Save metrics
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            method_name,
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params
        ])

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        model_path = f"models/{method_name}_ResNet_best.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, model_path)

        print("Best model saved!")

Trainable params: 51,813
Epoch 1/8
Train Loss: 2.8198, Train Acc: 0.3717
Val Loss:   2.2187, Val Acc:   0.4759
Time: 115.35s | Peak Mem: 980.27 MB
----------------------------------------
Best model saved!
Epoch 2/8
Train Loss: 2.0333, Train Acc: 0.5063
Val Loss:   2.0268, Val Acc:   0.5056
Time: 107.68s | Peak Mem: 980.27 MB
----------------------------------------
Best model saved!
Epoch 3/8
Train Loss: 1.8767, Train Acc: 0.5356
Val Loss:   1.9946, Val Acc:   0.5081
Time: 106.50s | Peak Mem: 980.27 MB
----------------------------------------
Best model saved!
Epoch 4/8
Train Loss: 1.7983, Train Acc: 0.5506
Val Loss:   1.9557, Val Acc:   0.5188
Time: 106.59s | Peak Mem: 980.27 MB
----------------------------------------
Best model saved!
Epoch 5/8
Train Loss: 1.7475, Train Acc: 0.5612
Val Loss:   1.9234, Val Acc:   0.5261
Time: 106.72s | Peak Mem: 980.27 MB
----------------------------------------
Best model saved!
Epoch 6/8
Train Loss: 1.7119, Train Acc: 0.5688
Val Loss:   1.9240, Va